In [ ]:
import os
# change working directory so that it picks up the grapehelper library
os.chdir('/home/ftorgano/rna-kg-analysis')
print(os.getcwd())

In [ ]:
from helper_lib import graph
from helper_lib import cache
from helper_lib import predict
import importlib
importlib.reload(graph)
importlib.reload(cache)
importlib.reload(predict)
cache.set_embedding_cache_dir("./RNA-KG_notebooks/Default_RNA-KG/cache/embeddings/")
import logging
logging.basicConfig(level=logging.INFO)
logging.getLogger().setLevel(logging.INFO)

In [ ]:
from grape import GraphVisualizer
from sklearn.metrics import balanced_accuracy_score
import pandas as pd
from itables import init_notebook_mode,show
init_notebook_mode(all_interactive=False)
import time
import numpy as np
import matplotlib.pyplot as plt
cycler_colors = ["#3f90da", "#ffa90e", "#bd1f01", "#94a4a2", "#832db6", "#a96b59", "#e76300", "#b9ac70", "#717581", "#92dadd"]
from cycler import cycler
plt.rcParams['axes.prop_cycle'] = cycler(color=cycler_colors)
df_formatters = {'balanced_acc_mean':'{:.2%}'.format,'balanced_acc_std':'{:.2%}'.format}

In [ ]:
undirected_rnakg = graph.load_rnakg_fixed()

# 10D embedding

## Edge Prediction Pipeline

In [ ]:
evaluation_schema = "Connected Monte Carlo"
edge_embedding_method = "Concatenate"
use_scale_free_distribution = True
training_unbalance_rate = 1/1
validation_unbalance_rate = 1/1

from grape.edge_prediction import DecisionTreeEdgePrediction, RandomForestEdgePrediction
from grape.edge_prediction import edge_prediction_evaluation
from grape.embedders import Node2VecSkipGramEnsmallen
# 856 minutes, only decision trees
# 310 minutes at 22:35
results_10 = pd.concat([
    edge_prediction_evaluation(
        graphs=undirected_rnakg,

        evaluation_schema = evaluation_schema,
        number_of_holdouts=5,
        holdouts_kwargs=dict(train_size=0.7),
        use_scale_free_distribution=use_scale_free_distribution,
        validation_unbalance_rates=validation_unbalance_rate,
        
        models=[
            DecisionTreeEdgePrediction(
                edge_embedding_methods = edge_embedding_method,
                use_scale_free_distribution = use_scale_free_distribution,
                training_unbalance_rate = training_unbalance_rate
            ),
            RandomForestEdgePrediction(
                edge_embedding_methods = edge_embedding_method,
                use_scale_free_distribution = use_scale_free_distribution,
                training_unbalance_rate = training_unbalance_rate,
                n_estimators=100,
                n_jobs=-1
            )
        ],
        node_features=Node2VecSkipGramEnsmallen(return_weight=5, explore_weight=0.2,embedding_size=10),
        smoke_test=False,
        enable_cache=True
    )
])

In [ ]:
results_10[["model_name","evaluation_mode","holdout_number","balanced_accuracy"]][results_10["evaluation_mode"]=="test"]

In [ ]:
results_10[["model_name","evaluation_mode","holdout_number","balanced_accuracy"]][results_10["evaluation_mode"]=="train"]

In [ ]:
test_results_tree_10 = results_10[(results_10["evaluation_mode"]=="test") & (results_10["model_name"]=="Decision Tree Classifier")][["model_name","evaluation_mode","holdout_number","balanced_accuracy"]]
print(test_results_tree_10["balanced_accuracy"].mean())
print(test_results_tree_10["balanced_accuracy"].std())

In [ ]:
test_results_tree_10 = results_10[(results_10["evaluation_mode"]=="test") & (results_10["model_name"]=="Decision Tree Classifier")][["model_name","evaluation_mode","holdout_number","balanced_accuracy"]]
print(test_results_tree_10["balanced_accuracy"].mean())
print(test_results_tree_10["balanced_accuracy"].std())

In [ ]:
test_results_forest_10 = results_10[(results_10["evaluation_mode"]=="test") & (results_10["model_name"]=="Random Forest Classifier")][["model_name","evaluation_mode","holdout_number","balanced_accuracy"]]
print(test_results_forest_10["balanced_accuracy"].mean())
print(test_results_forest_10["balanced_accuracy"].std())

In [ ]:
results_10.to_csv('./RNA-KG_notebooks/Default_RNA-KG/reports/report7/csv/edge_prediction_fullrnakg_dim10_unbiased.csv')